# Libs

In [12]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from collections import deque
import random

# Constants

In [13]:
# Параметры среды
env = gym.make("LunarLander-v2", continuous=True, render_mode="human")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
action_max = env.action_space.high[0]  # Максимальное значение действия

# Сетевые параметры
hidden_dim = 256
lr = 3e-4  # Скорость обучения
gamma = 0.99  # Коэффициент дисконтирования
alpha = 0.2  # Коэффициент энтропии
tau = 0.005  # Коэффициент для "soft" обновления целевой сети

# Буфер для хранения переходов (реплей буфер)
replay_buffer_size = 1000000
batch_size = 256


# Политика (Actor) — генерирует действия

In [14]:
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNet, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Linear(hidden_dim, action_dim)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        mean = self.mean(x)
        log_std = torch.clamp(self.log_std(x), min=-20, max=2)
        std = torch.exp(log_std)
        return mean, std

    def sample(self, state):
        mean, std = self.forward(state)
        normal = torch.distributions.Normal(mean, std)
        action = torch.tanh(normal.rsample())  # Стохастическое действие
        log_prob = normal.log_prob(action).sum(dim=-1, keepdim=True) - torch.log(1 - action.pow(2) + 1e-6).sum(dim=-1, keepdim=True)
        return action * action_max, log_prob
    

# Критики (Critic) — оценивают Q-функции

In [15]:
class QNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNet, self).__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


In [16]:
# Soft Q-Target Network (Double Q-learning)
def soft_update(net_target, net, tau):
    for param_target, param in zip(net_target.parameters(), net.parameters()):
        param_target.data.copy_(tau * param.data + (1 - tau) * param_target.data)
        

# Буфер воспроизведения

In [17]:
class ReplayBuffer:
    def __init__(self, max_size):
        self.buffer = deque(maxlen=max_size)

    def add(self, transition):
        self.buffer.append(transition)

    def sample(self, batch_size):
        transitions = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*transitions)
        return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(dones)

# Основная функция для обучения SAC

In [18]:
class SACAgent:
    def __init__(self, state_dim, action_dim):
        self.policy_net = PolicyNet(state_dim, action_dim)
        self.q_net1 = QNet(state_dim, action_dim)
        self.q_net2 = QNet(state_dim, action_dim)
        self.q_target_net1 = QNet(state_dim, action_dim)
        self.q_target_net2 = QNet(state_dim, action_dim)
        self.q_target_net1.load_state_dict(self.q_net1.state_dict())
        self.q_target_net2.load_state_dict(self.q_net2.state_dict())
        self.policy_optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.q_optimizer1 = optim.Adam(self.q_net1.parameters(), lr=lr)
        self.q_optimizer2 = optim.Adam(self.q_net2.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer(replay_buffer_size)

    def choose_action(self, state, evaluate=False):
        state = torch.FloatTensor(state).unsqueeze(0)
        action, _ = self.policy_net.sample(state)
        return action.detach().cpu().numpy()[0] if not evaluate else action_max * torch.tanh(self.policy_net(state)[0]).detach().cpu().numpy()[0]

    def update(self, batch_size):
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(batch_size)
        states = torch.FloatTensor(states)
        actions = torch.FloatTensor(actions)
        rewards = torch.FloatTensor(rewards).unsqueeze(1)
        next_states = torch.FloatTensor(next_states)
        dones = torch.FloatTensor(dones).unsqueeze(1)

        # Политика (Actor) обновляется для максимизации Q и энтропии
        new_actions, log_probs = self.policy_net.sample(states)
        q_value1 = self.q_net1(states, new_actions)
        q_value2 = self.q_net2(states, new_actions)
        q_min = torch.min(q_value1, q_value2) - alpha * log_probs
        policy_loss = (-q_min).mean()

        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        self.policy_optimizer.step()

        # Критики (Critic) обновляются для предсказания Q-функций
        next_actions, next_log_probs = self.policy_net.sample(next_states)
        target_q1 = self.q_target_net1(next_states, next_actions)
        target_q2 = self.q_target_net2(next_states, next_actions)
        target_q_min = torch.min(target_q1, target_q2) - alpha * next_log_probs
        target_q_value = rewards + gamma * (1 - dones) * target_q_min.detach()

        q1_loss = F.mse_loss(self.q_net1(states, actions), target_q_value)
        q2_loss = F.mse_loss(self.q_net2(states, actions), target_q_value)

        self.q_optimizer1.zero_grad()
        q1_loss.backward()
        self.q_optimizer1.step()

        self.q_optimizer2.zero_grad()
        q2_loss.backward()
        self.q_optimizer2.step()

        # Soft update целевых сетей
        soft_update(self.q_target_net1, self.q_net1, tau)
        soft_update(self.q_target_net2, self.q_net2, tau)




# Испаннцыыы!!!

In [19]:
# Тренировка агента
agent = SACAgent(state_dim, action_dim)

num_episodes = 1000
max_timesteps = 500

for episode in range(num_episodes):
    state = env.reset()[0]
    total_reward = 0

    for t in range(max_timesteps):
        action = agent.choose_action(state)
        next_state, reward, done, _, _ = env.step(action)
        agent.replay_buffer.add((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward

        if len(agent.replay_buffer.buffer) > batch_size:
            agent.update(batch_size)

        if done:
            break

    print(f"Episode {episode + 1}, Total Reward: {total_reward}")

Episode 1, Total Reward: -344.0655872174526
Episode 2, Total Reward: -419.1848802074246
Episode 3, Total Reward: -113.54731528923465
Episode 4, Total Reward: -647.0105425810718
Episode 5, Total Reward: -860.8271386567653
Episode 6, Total Reward: -1437.858249464596
Episode 7, Total Reward: -831.1354959832695
Episode 8, Total Reward: -1379.0518917239176
Episode 9, Total Reward: -1961.7727012982787
Episode 10, Total Reward: -1536.4565454385372
Episode 11, Total Reward: -1474.7426493312935
Episode 12, Total Reward: -1139.1654182211726
Episode 13, Total Reward: -789.8060673288544
Episode 14, Total Reward: -1622.6222733320826
Episode 15, Total Reward: -1143.842902536816
Episode 16, Total Reward: -792.9595858256761
Episode 17, Total Reward: -1227.7446786762707
Episode 18, Total Reward: -1740.5867275290218
Episode 19, Total Reward: -767.8482804912816
Episode 20, Total Reward: -1298.3851929361797
Episode 21, Total Reward: -1144.631042070039
Episode 22, Total Reward: -766.1098684434817
Episode 2

KeyboardInterrupt: 